In [1]:
import os
%pwd
os.chdir("../")
%pwd

'd:\\Data Science\\END to END Proj\\DrugToxicity'

In [2]:
## ENTITY
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    final_data_file: Path

In [4]:
from src.DrugToxicity.constant import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.DrugToxicity.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
            final_data_file=Path(config.final_data_file)
        )

        return data_ingestion_config

In [6]:
import os
import urllib.request as request
import zipfile
from src.DrugToxicity.utils.common import logger, get_size

In [7]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """Downloads zip from source_URL only if final_data_file does not exist."""
        if self.config.final_data_file.exists():
            logger.info(
                f"Dataset already exists at {self.config.final_data_file} "
                f"({get_size(self.config.final_data_file)}) — skipping download."
            )
            return

        os.makedirs(self.config.root_dir, exist_ok=True)

        if not self.config.local_data_file.exists():
            logger.info(f"Downloading from {self.config.source_URL} ...")
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"Download complete → {filename}")
            logger.debug(f"Headers: {headers}")
        else:
            logger.info(
                f"Zip already present at {self.config.local_data_file} "
                f"({get_size(self.config.local_data_file)}) — skipping download."
            )

    def extract_zip_file(self):
        """Extracts zip only if final_data_file does not exist."""
        if self.config.final_data_file.exists():
            logger.info("Extracted file already exists — skipping extraction.")
            return

        logger.info(
            f"Extracting {self.config.local_data_file} → {self.config.unzip_dir}"
        )
        os.makedirs(self.config.unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(self.config.unzip_dir)
        logger.info(f"Successfully extracted to {self.config.unzip_dir}")

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-26 23:56:16,338: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-26 23:56:16,341: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-26 23:56:16,345: INFO: common: created directory at: artifacts]
[2026-03-26 23:56:16,346: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-26 23:56:16,349: INFO: 2406810174: Downloading from https://github.com/gowtham-dd/Datasets/raw/main/tox21.zip ...]
[2026-03-26 23:56:18,140: INFO: 2406810174: Download complete → artifacts\data_ingestion\data.zip]
[2026-03-26 23:56:18,173: INFO: 2406810174: Extracting artifacts\data_ingestion\data.zip → artifacts\data_ingestion]
[2026-03-26 23:56:18,189: INFO: 2406810174: Successfully extracted to artifacts\data_ingestion]


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 62: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Dat